<a href="https://colab.research.google.com/github/NeoByteVoyager/PytorchLearning/blob/main/ML/Tatanic.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Tatanic生还预测

## 1.下载数据集

In [ ]:
# 下载 Kaggle 官方原汁原味的训练集（含 Survived 标签）
!wget https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv -O train.csv


--2026-07-03 03:34:54--  https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.108.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 60302 (59K) [text/plain]
Saving to: ‘train.csv’

train.csv           100%[===================>]  58.89K  --.-KB/s    in 0.007s  

2026-07-03 03:34:54 (7.83 MB/s) - ‘train.csv’ saved [60302/60302]



## 2.数据清洗
- 选择有用的特征

In [ ]:
import pandas as pd
# 1.读取文件
df = pd.read_csv("train.csv")
# 2.删除不必要的列
df.drop(columns=['Name','Ticket','Cabin','Embarked'],inplace=True)
# 3.修改后的csv重新写入文件
df.to_csv("cleaned_data.csv",index=False) # 要写上index = False


- 填充空白数据

In [ ]:
import pandas as pd
# 1.读取文件
df = pd.read_csv("cleaned_data.csv")
# 2.查看每一列的确实情况
print(df.isnull().sum())
# 3.计算Age列的中位数
age_median = df['Age'].median()
# 4.用中位数填补空缺值
df['Age'] = df['Age'].fillna(age_median)
# 5.写回文件
print(df.isnull().sum())
df.to_csv("cleaned_data.csv",index=False)

PassengerId      0
Survived         0
Pclass           0
Sex              0
Age            177
SibSp            0
Parch            0
Fare             0
dtype: int64
PassengerId    0
Survived       0
Pclass         0
Sex            0
Age            0
SibSp          0
Parch          0
Fare           0
dtype: int64


- 对性别进行编码

In [ ]:
import pandas as pd
# 1.读取文件
df = pd.read_csv("cleaned_data.csv")
print(df.info())
# 2.将性别转换为数字
if df['Sex'].dtype == 'object':
    df['Sex'] = df['Sex'].map({'female':0,'male':1})
df.to_csv("cleaned_data.csv",index=False)
print(df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Sex          891 non-null    object 
 4   Age          891 non-null    float64
 5   SibSp        891 non-null    int64  
 6   Parch        891 non-null    int64  
 7   Fare         891 non-null    float64
dtypes: float64(2), int64(5), object(1)
memory usage: 55.8+ KB
None
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Sex          891 non-null    int64  
 4   Age          891 non-null    float64
 5   SibSp

## 3. 构架Dataset和Dataloader

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
from  torch.utils.data import Dataset,DataLoader


# 1.数据集准备
df = pd.read_csv("cleaned_data.csv")

# 前700行做训练集，后面的做测试集
train_df = df.iloc[:700]
test_df = df.iloc[700:]

# 提取特征和标签
x_train = train_df.drop(columns=['PassengerId','Survived'], errors = 'ignore').values
y_train = train_df['Survived'].values

x_test = test_df.drop(columns=['PassengerId','Survived'], errors = 'ignore').values
y_test = test_df['Survived'].values


class MyData(Dataset):
    def __init__(self, features, labels):
        self.x_data = torch.tensor(features, dtype=torch.float32)
        self.y_data = torch.tensor(labels, dtype=torch.long)
        self.len = self.x_data.shape[0]

    def __getitem__(self, idx):
        return self.x_data[idx], self.y_data[idx]

    def __len__(self):
        return self.len

train_dataset = MyData(x_train, y_train)
test_dataset = MyData(x_test, y_test)

train_loader = DataLoader(dataset=train_dataset,
                          batch_size=32,
                          shuffle=True)
test_loader = DataLoader(dataset=test_dataset,
                         batch_size=32,
                         shuffle=False)

## 4.搭建模型

In [ ]:
# 2. 搭建模型
class MyModel(nn.Module):
    def __init__(self, in_features, out_features):
        super().__init__()
        self.layer1 = nn.Linear(in_features,24)
        self.layer2 = nn.Linear(24,12)
        self.layer3 = nn.Linear(12,out_features)
        self.sigmoid = torch.nn.Sigmoid()

    def forward(self, x):
        x=self.sigmoid(self.layer1(x))
        x=self.sigmoid(self.layer2(x))
        x=self.layer3(x)
        return x

# 实例化模型
model = MyModel(train_dataset.x_data.shape[1], 2)


## 5. 搭建损失函数和优化器

In [ ]:
# 3.损失函数和优化器
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.01, weight_decay=1e-4)


## 6.训练循环

In [ ]:
# 4.训练循环
for epoch in range(5000):
    total_loss = 0
    for idx, data in enumerate(train_loader):
        x, label = data
        # 前向传播
        y_hat = model(x)
        loss = criterion(y_hat, label)

        # 反向传播
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    if epoch % 10 == 0:
      print(f"epoch:{epoch}|loss:{total_loss:.4f}")
torch.save(model.state_dict(), "titanic_model.pth")
print("--> 完美！模型参数已成功保存为 'titanic_model.pth' \n")

epoch:0|loss:16.2338
epoch:10|loss:14.5295
epoch:20|loss:14.4274
epoch:30|loss:14.3169
epoch:40|loss:14.2070
epoch:50|loss:14.0878
epoch:60|loss:13.9760
epoch:70|loss:13.8639
epoch:80|loss:13.7615
epoch:90|loss:13.6803
epoch:100|loss:13.6231
epoch:110|loss:13.5710
epoch:120|loss:13.5471
epoch:130|loss:13.4952
epoch:140|loss:13.4786
epoch:150|loss:13.4451
epoch:160|loss:13.4375
epoch:170|loss:13.3957
epoch:180|loss:13.3885
epoch:190|loss:13.3786
epoch:200|loss:13.3705
epoch:210|loss:13.3566
epoch:220|loss:13.3359
epoch:230|loss:13.3145
epoch:240|loss:13.2937
epoch:250|loss:13.3161
epoch:260|loss:13.2848
epoch:270|loss:13.2575
epoch:280|loss:13.2586
epoch:290|loss:13.2356
epoch:300|loss:13.2358
epoch:310|loss:13.2164
epoch:320|loss:13.2071
epoch:330|loss:13.1931
epoch:340|loss:13.1837
epoch:350|loss:13.1803
epoch:360|loss:13.1418
epoch:370|loss:13.1249
epoch:380|loss:13.1258
epoch:390|loss:13.0784
epoch:400|loss:13.0471
epoch:410|loss:13.0209
epoch:420|loss:12.9859
epoch:430|loss:12.9715

## 7.测试集验证

In [ ]:
# 5.测试集验证
correct = 0
total = 0

with torch.no_grad():
    for data in test_loader:
        x, label = data
        y_hat = model(x)

        _, predict = torch.max(y_hat.data,1)

        total += x.size(0)
        correct += (predict == label).sum().item()

print(f"测试集上的准确率:{100 * correct / total:.2f}%")

测试集上的准确率:83.77%


## 8.恢复模型

In [ ]:
new_model = MyModel(in_features=train_dataset.x_data.shape[1], out_features=2)
new_model.load_state_dict(torch.load("titanic_model.pth"))
new_model.eval()

correct = 0
total = 0

with torch.no_grad():
    for data in test_loader:
        x, label = data
        y_hat = new_model(x)

        _, predict = torch.max(y_hat.data,1)

        total += x.size(0)
        correct += (predict == label).sum().item()

print(f"测试集上的准确率:{100 * correct / total:.2f}%")

测试集上的准确率:83.77%


## 9.预测test数据

In [ ]:
import pandas as pd
import torch

# 1. 读取好不容易下载好的真实测试集
raw_test_df = pd.read_csv("test.csv")

# =======================================================
# 💥 核心步骤：对测试集同步进行“数据清洗”（必须和训练集一模一样）
# =======================================================
# 复制一份，防止改乱原始数据
test_df_cleaned = raw_test_df.copy()

# (1) 删掉和训练集当时一样删掉的无关文本列 (Name, Ticket, Cabin, Embarked)
# 顺便删掉 PassengerId 和可能存在的 Unnamed: 0
columns_to_drop = ['PassengerId', 'Name', 'Ticket', 'Cabin', 'Embarked', 'Unnamed: 0']
test_df_cleaned.drop(columns=columns_to_drop, errors='ignore', inplace=True)

# (2) 处理缺失值：测试集的 Age 和 Fare 可能会有空缺，用中位数填满
test_df_cleaned['Age'] = test_df_cleaned['Age'].fillna(test_df_cleaned['Age'].median())
test_df_cleaned['Fare'] = test_df_cleaned['Fare'].fillna(test_df_cleaned['Fare'].median())

# (3) 文本转数字：把 Sex 列的 'male'->1, 'female'->0 (或者你训练集用的对应数字)
# ⚠️ 注意：这里的映射逻辑必须和你在训练集清洗时用的【完全一致】
test_df_cleaned['Sex'] = test_df_cleaned['Sex'].map({'male': 1, 'female': 0})

# =======================================================
# 2. 转换成 NumPy 数组和 PyTorch Tensor
# =======================================================
x_test_real = test_df_cleaned.values  # 这时候全部都是干净的数字了！
x_test_tensor = torch.tensor(x_test_real, dtype=torch.float32)

print("测试集清洗完成！当前特征形状为：", x_test_tensor.shape)
# 检查一下特征数是不是和训练集完全一样了？如果是，代表大门对上了！

# 3. 让模型进入评估模式，开始预测
new_model.eval()
submission_preds = []

with torch.no_grad():
    # 喂入模型前向传播
    outputs = new_model(x_test_tensor)

    # 提取得分最高的类别编号（0或1）
    _, predicted = torch.max(outputs.data, 1)
    submission_preds = predicted.tolist()

# 4. 按照 Kaggle 官方要求的格式组装并导出
submission_df = pd.DataFrame({
    "PassengerId": raw_test_df["PassengerId"],
    "Survived": submission_preds
})

submission_df.to_csv("submission.csv", index=False)
print("🎉 奇迹发生！Kaggle 提交文件 'submission.csv' 已成功生成，无任何报错！")

测试集清洗完成！当前特征形状为： torch.Size([418, 6])
🎉 奇迹发生！Kaggle 提交文件 'submission.csv' 已成功生成，无任何报错！
